In [1]:
%%writefile app.py
# ===== 설정 =====
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Titanic 대시보드", layout="wide")

# ===== 함수 =====
def filter_data(df, sexes, classes, age_range, ports):
    f = df[df["sex"].isin(sexes)
           & df["pclass"].isin(classes)
           & df["embarked"].isin(ports)]
    if age_range != (0, 80):
        f = f[f["age"].between(*age_range)]
    return f

def kpi(df):
    if len(df) == 0:
        return 0, 0, 0
    n = len(df)
    rate = round(df["survived"].mean() * 100, 1)
    age = round(df["age"].mean(), 1)
    return n, rate, age

def class_rate_bar(df):
    r = (df.groupby("pclass")["survived"]
           .mean().mul(100).round(1)
           .reset_index())
    fig = px.bar(r, x="pclass", y="survived", text="survived",
                 title="객실 등급별 생존율(%)")
    return fig

def age_hist(df):
    fig = px.histogram(df, x="age", color="survived", barmode="overlay",
                       opacity=0.6, title="생존 여부별 나이 분포")
    return fig

LABELS = {"sex": "성별", "class": "객실 등급", "embarked": "승선 항구"}

def group_rate_bar(df, col):
    r = (df.groupby(col)["survived"]
           .mean().mul(100).round(1)
           .reset_index())
    fig = px.bar(r, x=col, y="survived", text="survived",
                 title=f"{LABELS[col]}별 생존율(%)")
    return fig

def corr_heatmap(df):                                          # [추가] ② 상관 히트맵 함수
    corr = df.select_dtypes(include="number").corr().round(2)  # [추가] 수치형 열만 골라 상관계수
    fig = px.imshow(corr, text_auto=True,                      # [추가] 칸마다 값 표시
                    color_continuous_scale="RdBu_r",           # [추가] 양수=빨강, 음수=파랑
                    zmin=-1, zmax=1,                           # [추가] 색 범위 -1~1 고정
                    title="수치형 열 상관 히트맵")              # [추가]
    return fig                                                 # [추가]

def iqr_outliers(df, col):                                     # [추가] ③ IQR 이상치 개수 함수
    s = df[col].dropna()                                       # [추가] 결측 제외
    q1, q3 = s.quantile(0.25), s.quantile(0.75)                # [추가] 1사분위, 3사분위
    iqr = q3 - q1                                              # [추가]
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr                    # [추가] 정상 범위 경계
    count = int(((s < lo) | (s > hi)).sum())                   # [추가] 범위 밖 개수
    return count, lo, hi                                       # [추가]

# ===== 데이터 =====
@st.cache_data
def load_data():
    return pd.read_csv("titanic.csv")

df = load_data()

# ===== 제목 =====
st.title("타이타닉, 누가 살아남았나?")

# ===== 사이드바 =====
sexes = st.sidebar.multiselect("성별", ["male", "female"], ["male", "female"])
classes = st.sidebar.multiselect("객실 등급", [1, 2, 3], [1, 2, 3])
ages = st.sidebar.slider("나이", 0, 80, (0, 80))
ports = st.sidebar.multiselect("승선 항구", ["S", "C", "Q"], ["S", "C", "Q"])

if not sexes or not classes or not ports:
    st.warning("성별, 등급, 항구를 하나 이상 선택하세요.")
    st.stop()

# ===== 필터 =====
f = filter_data(df, sexes, classes, ages, ports)
if len(f) == 0:
    st.warning("조건에 맞는 승객이 없습니다.")
    st.stop()

# ===== 탭 =====                                               # [추가] ① 탭 3개
tab1, tab2, tab3 = st.tabs(["개요", "관계", "이상치"])          # [추가]

# ----- 개요 탭 -----
with tab1:                                                     # [추가] 기존 화면을 개요 탭 안으로
    n, rate, age = kpi(f)
    c1, c2, c3 = st.columns(3)
    c1.metric("승객 수", f"{n}명")
    c2.metric("생존율", f"{rate}%", f"{rate - 38.4:+.1f}%p")
    c3.metric("평균 나이", f"{age}세")

    left, right = st.columns(2)
    left.plotly_chart(class_rate_bar(f))
    right.plotly_chart(age_hist(f))

    col = st.selectbox("비교 기준",
                       ["sex", "class", "embarked"],
                       format_func=lambda c: LABELS[c])
    st.plotly_chart(group_rate_bar(f, col))

    with st.expander("데이터 보기"):
        st.dataframe(f)

# ----- 관계 탭 -----
with tab2:                                                     # [추가] ② 관계 탭
    st.plotly_chart(corr_heatmap(f))                           # [추가]
    st.caption("1에 가까우면 함께 커지고, -1에 가까우면 반대로 움직입니다.")  # [추가]

# ----- 이상치 탭 -----
with tab3:                                                     # [추가] ③ 이상치 탭
    num_cols = ["age", "fare", "sibsp", "parch"]               # [추가] 이상치 볼 수치형 열
    ocol = st.selectbox("열 선택", num_cols)                    # [추가] 열 선택
    count, lo, hi = iqr_outliers(f, ocol)                      # [추가]
    st.metric("IQR 이상치 수", f"{count}개")                    # [추가]
    st.caption(f"정상 범위: {lo:.1f} ~ {hi:.1f}")               # [추가]
    st.plotly_chart(px.box(f, y=ocol, points="outliers",       # [추가] 상자그림 + 이상치 점
                           title=f"{ocol} 상자그림"))           # [추가]

Writing app.py


In [4]:
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!streamlit run app.py &>/content/logs.txt & ./cloudflared tunnel --url http://localhost:8501

2026-11-24T02:17:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-24T02:17:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-24T02:17:43Z INF +--------------------------------------------------------------------------------------------+
2026-11-24T02:17:43Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-24T02:17:43Z INF |  https://encoding-citizen-quotations-joel.trycloudflar

In [3]:
import seaborn as sns
sns.load_dataset("titanic").to_csv("titanic.csv", index=False)